# Этап 2. Методы и модели (методичка 3.12)

Rule-based → нейросеть → гибрид → сравнение 3×3. Описание методов, формулы, допущения —
`docs/methods.md`.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU** (нужен для шага 2).
Сначала должен быть выполнен этап 1 (`01_stage1_data.ipynb`): нужны `thesis/data/processed/`.

Все результаты пишутся на Drive (`thesis/outputs/`), и каждый шаг пропускается, если его результат
уже есть. Если сессия Colab оборвалась — просто запустите ноутбук снова.

## 0. Настройка

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    token = userdata.get('GH_TOKEN')
    !git clone -q https://{token}@github.com/zhannaa0/kazakh-hybrid-nlp.git /content/project 2>/dev/null || git -C /content/project pull -q
    %cd /content/project
    !pip install -q -r requirements.txt
    PROJECT, THESIS = Path('/content/project'), Path('/content/drive/MyDrive/thesis')
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    THESIS = Path(os.environ.get('THESIS_DIR', PROJECT.parent / 'thesis'))
sys.path.insert(0, str(PROJECT))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите T4 для шага 2')
PREDS = THESIS / 'outputs' / 'preds'
SEEDS = [13, 42, 2026]

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Rule-based метод
Морфологический анализатор (аффиксы + словари) для POS и FEATS, шаблоны и газеттир для NER.
Работает на CPU за ~15 секунд.

In [ ]:
from src.rules import run as rules
if not (PREDS / 'test' / 'ner_rule.jsonl').exists():
    rules.run(THESIS)
!python -m src.eval.evaluate {PREDS}/dev --out {THESIS}/outputs/results_dev

## 2. Нейросеть
Один трансформер + головы задач. **UD** (POS + FEATS одной моделью): ~2–4 мин на seed на T4.
**NER** (KazNERD, 90 тыс. предложений): ~40 мин на seed — можно запускать по одному seed за сессию.

`MODEL` — открытое решение: сначала сравните два «хребта» на UD с одним seed (ячейка ниже),
затем зафиксируйте выбор в decision log.

In [ ]:
from src.neural.train import train

MODEL = 'xlm-roberta-base'          # или 'kz-transformers/kaz-roberta-conversational'

def done(task, seed):
    name = 'ner' if task == 'ner' else 'morph'
    return (PREDS / 'test' / f'{name}_neural_seed{seed}.jsonl').exists()

In [ ]:
# (необязательно) выбор «хребта»: оба на UD, seed 13, сравнение по dev
# for m in ['xlm-roberta-base', 'kz-transformers/kaz-roberta-conversational']:
#     log = train(THESIS, 'ud', m, seed=13)
#     print(m, log['best_dev_score'])

In [ ]:
for seed in SEEDS:                  # POS + морфология
    if not done('ud', seed):
        print(f'UD, seed {seed}'); train(THESIS, 'ud', MODEL, seed=seed)

In [ ]:
for seed in SEEDS:                  # NER — долго; при обрыве сессии запустите ячейку снова
    if not done('ner', seed):
        print(f'NER, seed {seed}'); train(THESIS, 'ner', MODEL, seed=seed)

## 3. Гибрид
Нейросеть → коррекция правилами. Параметры (λ для POS/FEATS; τ_p, γ, τ_g для NER) подбираются
на dev отдельно для каждого seed и применяются к test. Абляция — в `outputs/hybrid/params_seed*.json`.

In [ ]:
from src.hybrid import run as hybrid
for seed in SEEDS:
    hybrid.run(THESIS, seed)

In [ ]:
p = json.load(open(THESIS / 'outputs' / 'hybrid' / f'params_seed{SEEDS[0]}.json'))
for task, r in p.items():
    display(Markdown(f"**{task}** — параметры {r['params']}"))
    display(pd.Series({'rule': r['dev_rule'], 'neural': r['dev_neural'], 'hybrid': r['dev_hybrid'],
                       **r['ablation_dev']}, name='dev').to_frame())

## 4. Итоговое сравнение на test (методичка 7.7, 8.5, 8.7)
Таблица 3×3 (среднее ± std по seed), все попарные сравнения с 95% ДИ и поправкой Холма.

In [ ]:
RES = THESIS / 'outputs' / 'results_test'
!python -m src.eval.evaluate {PREDS}/test --out {RES} --significance

### Где помогает гибрид: известные и новые (OOV) слова

In [ ]:
from src.data.prepare import load_processed
from src.eval.breakdown import by_oov, train_vocabulary
from src.eval.io import read_predictions

proc = THESIS / 'data' / 'processed'
vocab = {'ner': train_vocabulary(load_processed(proc, 'kaznerd', ('train',))['train']),
         'pos': train_vocabulary(load_processed(proc, 'udktb', ('train',))['train'])}
vocab['morph'] = vocab['pos']
rows = []
for task in ('ner', 'pos', 'morph'):
    for method in ('rule', f'neural_seed{SEEDS[0]}', f'hybrid_seed{SEEDS[0]}'):
        path = PREDS / 'test' / f'{task}_{method}.jsonl'
        if path.exists():
            t = by_oov(read_predictions(path), task, vocab[task])
            col = 'f1' if task == 'ner' else 'score'
            rows.append({'task': task, 'method': method.split('_')[0],
                         'in_vocab': t.loc['in_vocab', col], 'oov': t.loc['oov', col]})
pd.DataFrame(rows).pivot(index='method', columns='task', values=['in_vocab', 'oov'])